In [1]:
from mpramnist.Fromel2025.dataset import FromelDataset
from mpramnist.Fromel2025.trainer import MaskedMSE
from mpramnist.Fromel2025.trainer import LitModel_Fromel

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t
import mpramnist.target_transforms as t_t

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

In [2]:
FromelDataset.HSPC_TARGETS

['State_1M',
 'State_2D',
 'State_3E',
 'State_4M',
 'State_5M',
 'State_6N',
 'State_7M']

In [3]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.AddFlanks("", FromelDataset.RIGHT_FLANK),  # this is original parameters for human_legnet
        t.RightCrop(245, 270),  # this is using for shifting
        t.LeftCrop(245, 245),
        t.ReverseComplement(0.5),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(FromelDataset.CONSTANT_LEFT_FLANK, FromelDataset.CONSTANT_RIGHT_FLANK),
        t.LeftCrop(245, 245),
        t.ReverseComplement(0),
        t.AddFeatureChannels(['batch']),
        t.Seq2Tensor(),
    ]
)

In [4]:
train_dataset = FromelDataset(split="train", transform=train_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")
val_dataset = FromelDataset(split="valid", transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root="../data/")

In [10]:
in_channels = len(train_dataset[0][0])
out_channels = len(train_dataset[0][1])

In [11]:
train_dl  = data.DataLoader(train_dataset, batch_size=1024, num_workers=64, shuffle=True)
val_dl  = data.DataLoader(val_dataset, batch_size=4096*2, num_workers=64, shuffle=False)

In [12]:
model = HumanLegNet(
                in_ch=in_channels,
                output_dim=out_channels,
                stem_ch=64,
                stem_ks=11,
                ef_ks=9,
                ef_block_sizes=[80, 96, 112, 128],
                pool_sizes=[2, 2, 2, 2],
                resize_factor=4,
        )
model.apply(initialize_weights)
        
seq_model = LitModel_Fromel(
    model=model, 
    activity_columns=FromelDataset.HSPC_TARGETS,
    loss=MaskedMSE(),
        weight_decay=2e-1,
        lr=0.005, 
    print_each=20,
)

In [16]:
trainer = L.Trainer(
            accelerator="gpu",
            devices=[0],
            max_epochs=20,
            gradient_clip_val=1,
            precision="16-mixed",
            enable_progress_bar=True,
            num_sanity_val_steps=0,
        )
trainer.fit(seq_model, 
            train_dataloaders=train_dl,
            val_dataloaders=val_dl)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.

  | Name          | Type                  | Params | Mode 
----------------------------------------------------------------
0 | model         | HumanLegNet           | 1.3 M  | train
1 | loss          | MaskedMSE             | 0 

Epoch 19: 100%|██████████| 45/45 [00:06<00:00,  6.94it/s, v_num=3, train_loss_step=0.0563, val_loss=0.0771, val_State_1M_pearson=0.623, val_State_2D_pearson=0.594, val_State_3E_pearson=0.571, val_State_4M_pearson=0.639, val_State_5M_pearson=0.602, val_State_6N_pearson=0.498, val_State_7M_pearson=0.527, val_pearson=0.579, train_loss_epoch=0.0612]
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| Epoch: 19 | Val Loss: 0.07770 
| Val Pearson State_1M: 0.62105 | Val Pearson State_2D: 0.59209 | Val Pearson State_3E: 0.56975 | Val Pearson State_4M: 0.63743 | Val Pearson State_5M: 0.60091 | Val Pearson State_6N: 0.49554 | Val Pearson State_7M: 0.52549 | Mean Val Pearson: 0.57747 |
| Train Pearson State_1M: 0.71208 | Train Pearson State_2D: 0.70379 | Train Pearson State_3E: 0.

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 45/45 [00:10<00:00,  4.10it/s, v_num=3, train_loss_step=0.0563, val_loss=0.0777, val_State_1M_pearson=0.621, val_State_2D_pearson=0.592, val_State_3E_pearson=0.570, val_State_4M_pearson=0.637, val_State_5M_pearson=0.601, val_State_6N_pearson=0.496, val_State_7M_pearson=0.525, val_pearson=0.577, train_loss_epoch=0.0605]


In [19]:
test_dataset = FromelDataset(split='test', transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dl  = data.DataLoader(test_dataset, batch_size=4096*2, num_workers=64, shuffle=False)

In [20]:
trainer.test(seq_model, dataloaders=test_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  test_State_1M_pearson     0.6542006731033325
  test_State_2D_pearson     0.6039196848869324
  test_State_3E_pearson     0.5522845387458801
  test_State_4M_pearson     0.6649710536003113
  test_State_5M_pearson     0.6167681813240051
  test_State_6N_pearson      0.505246102809906
  test_State_7M_pearson     0.5486388206481934
        test_loss           0.08119426667690277
      test_pearson          0.5922898650169373
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.08119426667690277,
  'test_State_1M_pearson': 0.6542006731033325,
  'test_State_2D_pearson': 0.6039196848869324,
  'test_State_3E_pearson': 0.5522845387458801,
  'test_State_4M_pearson': 0.6649710536003113,
  'test_State_5M_pearson': 0.6167681813240051,
  'test_State_6N_pearson': 0.505246102809906,
  'test_State_7M_pearson': 0.5486388206481934,
  'test_pearson': 0.5922898650169373}]

In [21]:
test_dataset = FromelDataset(split='genome', transform=test_transform, targets=FromelDataset.HSPC_TARGETS, root = "../data")
test_dl  = data.DataLoader(test_dataset, batch_size=4096*2, num_workers=64, shuffle=False)
trainer.test(seq_model, dataloaders=test_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 27.69it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  test_State_1M_pearson     0.02586011029779911
  test_State_2D_pearson     0.05976604297757149
  test_State_3E_pearson     0.04884720966219902
  test_State_4M_pearson    0.057854779064655304
  test_State_5M_pearson             nan
  test_State_6N_pearson     0.0973740965127945
  test_State_7M_pearson    0.043420080095529556
        test_loss           0.13997191190719604
      test_pearson          0.05552038550376892
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.13997191190719604,
  'test_State_1M_pearson': 0.02586011029779911,
  'test_State_2D_pearson': 0.05976604297757149,
  'test_State_3E_pearson': 0.04884720966219902,
  'test_State_4M_pearson': 0.057854779064655304,
  'test_State_5M_pearson': nan,
  'test_State_6N_pearson': 0.0973740965127945,
  'test_State_7M_pearson': 0.043420080095529556,
  'test_pearson': 0.05552038550376892}]